# 00 Data Ingestion

This notebook pulls the two primary datasets used throughout this project:

| # | Source | What it contains |
|---|--------|------------------|
| 1 | Kaggle `robikscube/hourly-energy-consumption` | PJM hourly load (MWh) by region, 2002–2018 |
| 2 | EIA / ICE `ice_electric-*.xls(x)` | PJM West wholesale day-ahead spot prices ($/MWh), 2014–2025 |

All paths are relative to the **repo root**. This notebook will run from the repo root as the working directory.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# python-dotenv searches the current directory and all parents,
# so this finds .env at the repo root regardless of CWD.
load_dotenv()

# Change working directory to repo root (notebook is in notebooks/ subfolder)
os.chdir("..")

RAW_DIR = Path("data/raw")

---
## 1  PJM Hourly Energy Consumption — Kaggle

**Dataset:** [`robikscube/hourly-energy-consumption`](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption)  
**What it is:** Hourly electricity consumption (MWh) for regions within the PJM Interconnection, sourced from PJM's public data portal. Covers roughly 2002–2018 for multiple sub-regions (AEP, COMED, PJME, PJMW, etc.).  
**Why we need it:** Provides the demand-side signal. `PJMW_hourly.csv` is the PJM West region load series that pairs directly with the spot-price data in section 2.

Credentials are read from `.env` at the repo root (keys `KAGGLE_USERNAME` and `KAGGLE_KEY`). They are never hardcoded.

In [2]:
# Inject credentials into the environment so the Kaggle client picks them up.
kaggle_user = os.getenv("KAGGLE_USERNAME")
kaggle_key  = os.getenv("KAGGLE_KEY")

if not kaggle_user or not kaggle_key:
    raise EnvironmentError(
        "KAGGLE_USERNAME and KAGGLE_KEY must be set in .env at the repo root."
    )

os.environ["KAGGLE_USERNAME"] = kaggle_user
os.environ["KAGGLE_KEY"]      = kaggle_key

# Import after env vars are set so the client authenticates correctly.
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

DATASET_SLUG = "robikscube/hourly-energy-consumption"
PJMW_CSV     = RAW_DIR / "PJMW_hourly.csv"

if not PJMW_CSV.exists():
    print(f"Downloading {DATASET_SLUG} -> {RAW_DIR} ...")
    api.dataset_download_files(DATASET_SLUG, path=RAW_DIR, unzip=True)
    print("Download complete.")
else:
    print(f"Dataset already present at {PJMW_CSV}, skipping download.")

df_load = pd.read_csv(
    PJMW_CSV,
    parse_dates=["Datetime"],
    index_col="Datetime",
)

print(f"\nLoaded PJMW_hourly: {df_load.shape[0]:,} rows x {df_load.shape[1]} columns")

Dataset already present at data/raw/PJMW_hourly.csv, skipping download.

Loaded PJMW_hourly: 143,206 rows x 1 columns


In [3]:
# Clean up column names
df_load.index.name = "datetime"

df_load.rename(columns={"PJMW_MW": "consumption_mw"}, inplace=True)

### Ensure correct column types

In [4]:
df_load.index = pd.to_datetime(df_load.index)

df_load = df_load.astype({
    "consumption_mw": "float32"
    })

# Sort index column ascending
df_load.sort_index(inplace=True)

---
## 2  PJM West Wholesale Spot Prices — EIA / ICE

**Files:** `data/raw/ice_electric-<year>final.xls(x)`, 2014–2018  
**What they are:** Daily wholesale electricity spot-price reports published by ICE (Intercontinental Exchange) and mirrored by EIA. Each file covers one calendar year and contains settlements for multiple North American power hubs (ERCOT, PJM West, Mass Hub, etc.).  
**Why we need it:** The PJM West Hub price is the clearing price for the PJM West market — the target variable we will forecast.

The cell below globs for every `ice_electric*.xls` and `ice_electric*.xlsx` file directly inside `data/raw/`, reads each into a DataFrame, and concatenates them.

In [5]:
# Function to load and concatenate multiple Excel files into a single DataFrame
def load_and_concat_excel_files(file_paths):
    """
    Load multiple Excel files and concatenate them into a single DataFrame.
    
    Args:
        file_paths: List of Path objects or strings pointing to .xls or .xlsx files
    
    Returns:
        Concatenated DataFrame with a '_source_file' column, or empty DataFrame if no files found
    """
    frames = []
    
    for filepath in file_paths:
        engine = "xlrd" if str(filepath).endswith(".xls") else "openpyxl"
        df_yr = pd.read_excel(filepath, engine=engine, na_values=["", "NA", "N/A", "na", "n/a"])
        
        # Normalise column names to lowercase-underscore BEFORE concatenation.
        # This will ensure consistent column names, so we can append all the files together.
        df_yr.columns = (
            df_yr.columns
            .str.strip()
            .str.lower()
            .str.replace(r"[\s/]+", "_", regex=True)
            .str.replace(r"[^\w]", "", regex=True)
            .str.replace(r"__", "_", regex=True)
        )
        
        df_yr["_source_file"] = Path(filepath).name
        frames.append(df_yr)
    
    if frames:
        df = pd.concat(frames, ignore_index=True)
    else:
        print("No Excel files found to load.")
        df = pd.DataFrame()
    
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

Let's combine all .xls and .xlsx files into one data frame.

In [6]:
historical_xls_files  = sorted(RAW_DIR.glob("ice_electric-historical/*.xls"))
xls_files  = sorted(RAW_DIR.glob("ice_electric*.xls"))
xlsx_files = sorted(RAW_DIR.glob("ice_electric*.xlsx"))

all_price_files = historical_xls_files + xls_files + xlsx_files

df_prices = load_and_concat_excel_files(all_price_files)

Shape: 27,101 rows x 13 columns


Now we'll filter specifically for PJM, as this is the scope of the project.

Then we'll sort by the `trade_date`, which is the date qualifying transactions were executed on ICE and the weighted average price was set. This is when the market priced the electricity a buyer such as a data centre operator would have paid. Source: https://www.eia.gov/electricity/wholesale

In [7]:
# We'll filter the data only to include PJM prices
df_prices = df_prices[df_prices["price_hub"].str.match(r"^PJM.*$", na=False)]

# We'll finally sort the data by trade_date and reset the index.
df_prices = df_prices.sort_values("trade_date").reset_index(drop=True)

### Ensure correct column types

In [8]:
# Remove commas and convert to integer
df_prices['daily_volume_mwh'] = df_prices['daily_volume_mwh'].astype(str).str.replace(',', '').str.strip().astype('Int64')
df_prices['number_of_trades'] = df_prices['number_of_trades'].astype(str).str.replace(',', '').str.strip().astype('Int64')

In [9]:
# Ensure all the correct column types for the df_prices dataframe.
df_prices = df_prices.astype({
    "price_hub": "string",
    "trade_date": "datetime64[ns]",
    "delivery_start_date": "datetime64[ns]",
    "delivery_end_date": "datetime64[ns]",
    "high_price_mwh": "float64",
    "low_price_mwh": "float64",
    "wtd_avg_price_mwh": "float64",
    "change": "float64",
    "daily_volume_mwh": "Int64",
    "number_of_trades": "Int64",
    "number_of_counterparties": "Int64",
    "_source_file": "string"
})

---
## 3 Write processed data as parquet

In [10]:
# Write the cleaned dataframes as parquet files to the processed/ directory for use in later notebooks.
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)
df_load.to_parquet(PROCESSED_DIR / "pjm_hourly_energy_consumption.parquet", index=True)
df_prices.to_parquet(PROCESSED_DIR / "ice_electric_prices.parquet", index=False)